# Bouw je eigen beeldclassifier

Vorige week gebruikte je **Teachable Machine**: webcam aanzetten, op "train"
klikken, en klaar was je classifier. Het werkte in een paar seconden, met
bijna geen data. Vandaag bouw je datzelfde trucje zelf, in code, en zie je
precies waarom het werkt.

Het korte antwoord: Teachable Machine leert eigenlijk niet vanaf nul kijken.
Het leent een netwerk dat al geleerd heeft te kijken met **1,4 miljoen**
foto's (ImageNet), en traint alleen een klein laagje bovenop voor *jouw*
klassen. Dat heet **transfer learning**, en dat is precies waarom een paar
tientallen foto's en een paar minuten genoeg zijn.

Deze keer is niks verborgen:
- **Jij** kiest de klassen.
- **Jij** verzamelt de afbeeldingen (een live zoekopdracht, geen kant-en-klare
  dataset die iemand anders je gaf).
- **Jij** kunt elke regel lezen die het model traint en een voorspelling doet.

Dat verschil doet ertoe. Een model dat je ergens download is een black box
tenzij je zelf gaat kijken — je weet niet met welke data het getraind is of
wat er precies in het bestand zit. Een model dat je zelf bouwt, van
afbeeldingen die je zelf ziet, is dat niet. Stap 9 borduurt hier precies op
voort, met dezelfde pipeline voor iets minder comfortabels dan hondenrassen.

## Stap 0 — Setup

Voer dit één keer uit. `tensorflow` en `matplotlib` bestaan al in Colab —
alleen `ddgs` (afbeeldingen zoeken) en `gradio` (de live demo aan het eind)
moeten nog geïnstalleerd worden.

In [ ]:
!pip install -q ddgs gradio

import io
import shutil
from pathlib import Path

import numpy as np
import requests
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from ddgs import DDGS

print("TensorFlow:", tf.__version__)

## Stap 1 — Kies je klassen

Kies 2-4 dingen die een camera uit elkaar kan houden. Goede keuzes zijn
visueel duidelijk verschillend, maar niet *triviaal* verschillend — twee
hondenrassen, twee stijlen stoelen, twee van je eigen handgebaren. Vermijd
losse dubbelzinnige woorden (`"muis"` levert je zowel dieren als
computermuizen op).

De sleutel (key) van de dictionary wordt de map-/klassennaam (geen
spaties). De waarde (value) is de daadwerkelijke zoekterm — een woord als
`"photo"` toevoegen filtert meestal cartoons/logo's eruit en levert echte
foto's op. Tip: Engelse zoektermen leveren vaak meer en betere resultaten op
dan Nederlandse. Er is deze keer geen voorbeeld gegeven — schrijf je eigen
versie.

In [ ]:
# TODO: voeg 2-4 items toe. Key = mapnaam (geen spaties), value = zoekterm.
# Vorm (schrijf je eigen waarden, kopieer dit niet): "cat": "cat photo"
CLASSES = {

}

assert CLASSES, "Voeg minstens 2 klassen toe aan CLASSES hierboven voordat je verdergaat"

IMAGES_PER_CLASS = 100
DATA_DIR = Path("data")

## Stap 2 — Verzamel de afbeeldingen

Dit doorzoekt DuckDuckGo-afbeeldingen voor elke klasse en downloadt de eerste
`IMAGES_PER_CLASS` resultaten, en slaat elke afbeelding opnieuw op als een
schone RGB-JPEG (sommige zoekresultaten zijn kapotte links, vreemde
formaten, of corrupt — die worden stilletjes overgeslagen, waardoor je
uiteindelijk minder afbeeldingen kunt krijgen dan je vroeg).

**Verwacht wat rommel in de resultaten.** Dat is geen bug — daar ga je in
Stap 3 mee aan de slag. Het is ook een kleine, laagdrempelige preview van het
thema van Stap 9: je kunt niet volledig vertrouwen op wat je niet zelf hebt
gecontroleerd.

`scrape_class` hieronder is al gegeven — het is loodgieterswerk
(HTTP-requests, foutafhandeling), niet het interessante deel. De loop die
het daadwerkelijk gebruikt, direct erna, is aan jou.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}

def scrape_class(query, out_dir, n):
    out_dir.mkdir(parents=True, exist_ok=True)
    with DDGS() as ddgs:
        results = list(ddgs.images(query, max_results=n))
    saved = 0
    for r in results:
        url = r.get("image")
        if not url:
            continue
        try:
            resp = requests.get(url, headers=HEADERS, timeout=6)
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            img.save(out_dir / f"{saved:03d}.jpg", "JPEG", quality=90)
            saved += 1
        except Exception:
            continue
    return saved

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

# TODO: roep voor elk (folder_name, query) paar in CLASSES de functie
# scrape_class(query, DATA_DIR / folder_name, IMAGES_PER_CLASS) aan, sla het
# resultaat op in n, en print hoeveel afbeeldingen die klasse echt kreeg.
for folder_name, query in CLASSES.items():
    n = ___
    print(f"{folder_name}: {n} afbeeldingen opgeslagen")

## Stap 3 — Bekijk wat je eigenlijk hebt gekregen

Voordat je erop traint, bekijk het eerst — hetzelfde instinct als het
controleren van een gedownload model voordat je het vertrouwt. Dit rooster
toont een willekeurige steekproef uit elke klasse.

In [ ]:
def show_samples(data_dir, n=6):
    class_dirs = sorted(d for d in Path(data_dir).iterdir() if d.is_dir())
    fig, axes = plt.subplots(len(class_dirs), n, figsize=(n * 2, len(class_dirs) * 2))
    for row, cdir in enumerate(class_dirs):
        files = list(cdir.glob("*.jpg"))
        sample = np.random.choice(files, size=min(n, len(files)), replace=False)
        for col in range(n):
            ax = axes[row][col] if len(class_dirs) > 1 else axes[col]
            ax.axis("off")
            if col < len(sample):
                ax.imshow(Image.open(sample[col]))
                if col == 0:
                    ax.set_title(cdir.name, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

show_samples(DATA_DIR)

Zie je rommel (verkeerd onderwerp, een logo, een screenshot, een diagram in
plaats van een foto)? Verwijder het. Makkelijkste manier: open het
**Files**-paneel links, zoek het bestand onder `data/<klassennaam>/`,
rechtsklik → verwijderen. Of noteer de slechte bestanden hieronder en voer
de cel uit — in beide gevallen: voer daarna het voorbeeldrooster hierboven
opnieuw uit om te controleren.

In [ ]:
# Voorbeeld: BAD_FILES = ["golden_retriever/013.jpg", "poodle/047.jpg"]
BAD_FILES = []

for f in BAD_FILES:
    (DATA_DIR / f).unlink(missing_ok=True)

print(f"{len(BAD_FILES)} bestand(en) verwijderd")

## Stap 4 — Maak van de mappen een dataset

Standaard Keras-hulpfunctie: wijs hem naar een map met
`klassennaam/afbeelding.jpg`-mappen, en hij herkent zelf de klassen en zet
20% opzij voor validatie (afbeeldingen waar het model nooit op traint,
gebruikt om te controleren of het echt generaliseert).

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Klassen:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## Stap 5 — Leen een paar ogen

Dit is de eigenlijke truc achter Teachable Machine. `MobileNetV2` hieronder
weet al hoe het moet "kijken" — het is getraind op 1,4 miljoen
ImageNet-foto's om 1000 categorieën uit elkaar te houden. We bevriezen het
(`trainable = False`, zodat die kennis niet overschreven wordt) en trainen
alleen een klein nieuw kopje bovenop voor *jouw* klassen.

`data_augmentation` flipt/roteert/zoomt trainingsafbeeldingen willekeurig
bij elke epoch — een goedkope manier om meer variatie uit ~100 foto's te
halen. Hetzelfde idee (augmentatie is belangrijk bij kleine datasets) komt
ook terug in het ESP32-cijferherkenningsproject, als je die repo later
bekijkt.

Twee dingen hieronder zijn aan jou:
- Moet `base_model.trainable` `True` of `False` zijn? (Willen we dat de
  kennis uit 1,4 miljoen foto's blijft veranderen tijdens het trainen, of
  moet die vaststaan?)
- De laatste laag moet één score per klasse geven, omgezet naar kansen die
  optellen tot 1 — die activatie heet `softmax`. Hoeveel outputs heeft die
  laag nodig?

In [ ]:
num_classes = len(class_names)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = ___  # True of False?

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(___, activation="softmax")(x)  # hoeveel outputs?
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Stap 6 — Train

Met ~100 afbeeldingen per klasse zou dit ruim onder een minuut per epoch
moeten duren op Colab's CPU. Kies hieronder een aantal epochs (volledige
doorlopen van de trainingsdata) — probeer iets tussen 5 en 15. Meer epochs
betekent meer trainingstijd, niet automatisch meer nauwkeurigheid; de
grafiek na het trainen laat zien of het de moeite waard was.

In [ ]:
EPOCHS = ___  # kies een getal, bijv. tussen 5 en 15

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("epoch")
plt.legend()
plt.title("Hoe snel leerde het?")
plt.show()

## Stap 7 — Waar gaat het mis?

De interessante gevallen zijn de fouten, niet de juiste voorspellingen. Vaak
zijn ze direct te herleiden tot een slechte foto die je in Stap 3 over het
hoofd zag — verkeerd gelabeld, verkeerd onderwerp, of gewoon echt
dubbelzinnig.

In [ ]:
wrong = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)
    for img, true, pred in zip(images, labels.numpy(), pred_labels):
        if true != pred:
            wrong.append((img.numpy().astype("uint8"), class_names[true], class_names[pred]))

print(f"{len(wrong)} fout(en) in de validatieset")

n_show = min(6, len(wrong))
if n_show:
    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2, 2))
    axes = [axes] if n_show == 1 else axes
    for i in range(n_show):
        img, true, pred = wrong[i]
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"echt: {true}\nraadde: {pred}", fontsize=8)
    plt.tight_layout()
    plt.show()

## Stap 8 — Probeer het live

Richt je webcam op iets, of upload een foto, en kijk hoe je eigen model
raadt.

In [ ]:
import gradio as gr

def predict(img):
    if img is None:
        return {}
    img = Image.fromarray(img).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img), axis=0).astype("float32")
    preds = model.predict(arr, verbose=0)[0]
    return {class_names[i]: float(preds[i]) for i in range(len(class_names))}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(sources=["webcam", "upload"], type="numpy"),
    outputs=gr.Label(num_top_classes=num_classes),
    title="Jouw Classifier",
)
demo.launch(debug=True)

## Stap 9 — Opdracht: Dark Technology

De pipeline die je net gebruikte — afbeeldingen zoeken, een classifier
trainen, live zetten — is ook precies hoe echte, schadelijke classifiers
gebouwd worden. Een bekend voorbeeld: in 2015 labelde het automatische
tag-systeem van Google Photos foto's van zwarte mensen als gorilla's.
Niemand had de bedoeling om dat te bouwen — het kwam voort uit wat er níét
in de trainingsdata zat. Niet de tool was het probleem, maar de data, en wat
er met de output gedaan werd.

**Jouw opdracht:** bedenk je eigen idee voor iets wat een classifier zou
kunnen doen dat telt als *Dark Technology* — een echte toepassing die je met
wat je nu weet daadwerkelijk zou kunnen bouwen, en die ongemakkelijk zou
moeten voelen als het echt uitgebracht werd. Het hoeft niet op het
Google-voorbeeld te lijken. Denk aan surveillance, profilering, uitsluiting,
het afleiden van iets privés uit een afbeelding, of een classifier die
alleen goed werkt voor één groep mensen. Het gaat erom: wat maakt jouw idee
"dark," en waarom is het zo makkelijk om te bouwen?

Dan ga je het **bouwen**. Kopieer dit notebook, vervang je eigen klassen en
zoektermen, en zorg dat het écht end-to-end werkt — via de live demo in Stap
8. Het hoeft niet gepolijst te zijn. Het moet werken.

Je laat het straks aan de klas zien — 2-3 minuten per persoon/groep:
- Wat heb je gebouwd?
- Waarom telt het als Dark Technology?
- Wat zou er anders moeten zijn aan de data, of het proces, om dit in het
  echt te voorkomen?

Geen idee? Wat startpunten:
- Een classifier die iets over een persoon afleidt dat diegene niet zelf
  wilde laten zien.
- Een classifier getraind op een smalle groep mensen, die daarna op
  iedereen wordt losgelaten.
- Een classifier waarvan de fouten de ene groep veel harder raken dan de
  andere.

## Extra opdrachten (als je vroeg klaar bent)

- Voeg een 3e of 4e klasse toe.
- Probeer `MobileNetV3Small` of `EfficientNetB0` in plaats van
  `MobileNetV2` — verandert de nauwkeurigheid of snelheid?
- Ontdooi (`unfreeze`) de laatste paar lagen van `base_model` en fine-tune
  met een heel lage learning rate (`1e-5`) voor een paar extra epochs —
  meestal een kleine verbetering in nauwkeurigheid, ten koste van veel
  langzamer trainen.
- `model.save("my_model.keras")` en download het — de
  ESP32-cijferherkenningsrepo (als je instructeur die deelt) laat zien wat
  de *volgende* stap eruitziet: een Keras-model verkleinen zodat het
  volledig op een microcontroller van 5 dollar draait, zonder cloud.